# Практика 13. Точкові оцінки та перевірка гіпотез: обчислення p-value, інтерпретація результату

**Варіант 1: Київ**

Мета: перевірити, чи узгоджується вибірка поточних липневих температур Києва зі старою нормою `μ₀ = 20.0 °C`. Вибірка `daily_temps` є єдиною для всіх завдань.

In [1]:
import numpy as np
from scipy import stats

variant = 1
city = 'Київ'
july_mean = 21.0
mu0 = 20.0
alpha = 0.05

np.random.seed(variant)
daily_temps = np.random.normal(loc=july_mean, scale=2.5, size=30)

print(f'Місто: {city}')
print(f'Варіант: {variant}')
print(f'Стара норма μ₀: {mu0:.1f} °C')
print('daily_temps =', daily_temps)

Місто: Київ
Варіант: 1
Стара норма μ₀: 20.0 °C
daily_temps = [25.06086341 19.47060897 19.67957062 18.31757844 23.16351907 15.24615326
 25.36202941 19.09698275 21.79759774 20.37657406 24.65526984 15.84964823
 20.19395699 20.03986411 23.83442361 18.25027183 20.56892948 18.80535396
 21.10553437 22.45703803 18.24845206 23.86180927 23.2539768  22.25623585
 23.25213987 19.29068035 20.69277444 18.66057641 20.3302798  22.32588867]


## Завдання 1. Вибірка і точкові оцінки

Обчислюємо вибіркове середнє `x̄` і вибіркове стандартне відхилення `s`. Вони є точковими оцінками середнього та стандартного відхилення генеральної сукупності.

In [2]:
n = len(daily_temps)
x_bar = daily_temps.mean()
s = daily_temps.std(ddof=1)

print(f'n = {n}')
print(f'x̄ = {x_bar:.4f} °C')
print(f's = {s:.4f} °C')
print(f'x̄ - μ₀ = {x_bar - mu0:.4f} °C')

n = 30
x̄ = 20.8502 °C
s = 2.5659 °C
x̄ - μ₀ = 0.8502 °C


## Завдання 2. Формулювання H0 і H1

Питання «стара норма більше не відповідає дійсності» допускає зміну і в бік потепління, і в бік похолодання. Тому тип тесту обираємо **до** обчислення результату як двобічний.

- **H₀:** `μ = μ₀ = 20.0 °C`; стара норма для Києва правильна.
- **H₁:** `μ ≠ μ₀`; середня липнева температура відрізняється від старої норми.

Двобічний тест заздалегідь враховує відхилення в обидва боки.

## Завдання 3. z-статистика і p-value

За умовою практики використовуємо стандартну похибку `se = s / √n` і `scipy.stats.norm.cdf()`. Для двобічного тесту p-value враховує обидва хвости.

In [3]:
se = s / np.sqrt(n)
z = (x_bar - mu0) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print(f'se = {se:.4f} °C')
print(f'z = {z:.4f}')
print(f'p-value = {p_value:.4f}')

se = 0.4685 °C
z = 1.8147
p-value = 0.0696


## Завдання 4. Рішення при α = 0.05 і висновок

Якщо `p-value < α`, відхиляємо H₀; інакше не відхиляємо H₀.

In [4]:
reject_h0 = p_value < alpha
decision = 'відхиляємо H₀' if reject_h0 else 'не відхиляємо H₀'
direction = 'вища' if x_bar > mu0 else 'нижча'

print(f'p-value = {p_value:.4f}, α = {alpha:.2f}')
print(f'Рішення: {decision}.')
if reject_h0:
    print(f'Висновок: дані дають статистично значущі підстави вважати, що середня липнева температура у місті {city} відрізняється від старої норми {mu0:.1f} °C; вибіркове середнє {direction} за норму (p = {p_value:.4f}, двобічний тест, α = {alpha:.2f}).')
else:
    print(f'Висновок: дані не дають статистично значущих підстав вважати, що середня липнева температура у місті {city} відрізняється від старої норми {mu0:.1f} °C (p = {p_value:.4f}, двобічний тест, α = {alpha:.2f}).')

p-value = 0.0696, α = 0.05
Рішення: не відхиляємо H₀.
Висновок: дані не дають статистично значущих підстав вважати, що середня липнева температура у місті Київ відрізняється від старої норми 20.0 °C (p = 0.0696, двобічний тест, α = 0.05).


## Завдання 5. 95%-й довірчий інтервал і помилки I/II роду

Для порівняння з тестом будуємо інтервал за формулою з методички: `x̄ ± 1.96 · se`. Для двобічного тесту при `α = 0.05` стара норма потрапляє в інтервал тоді й лише тоді, коли H₀ не відхиляється.

In [5]:
z_critical = stats.norm.ppf(1 - alpha / 2)
ci_low = x_bar - z_critical * se
ci_high = x_bar + z_critical * se
mu0_in_ci = ci_low <= mu0 <= ci_high

print(f'Критичне значення z₀.₉₇₅ = {z_critical:.4f}')
print(f'95%-й ДІ для μ: ({ci_low:.4f}, {ci_high:.4f}) °C')
print(f'Чи потрапляє μ₀ = {mu0:.1f} °C в інтервал? {mu0_in_ci}')
print(f'Висновок через ДІ: {"не відхиляємо H₀" if mu0_in_ci else "відхиляємо H₀"}.')

Критичне значення z₀.₉₇₅ = 1.9600
95%-й ДІ для μ: (19.9320, 21.7683) °C
Чи потрапляє μ₀ = 20.0 °C в інтервал? True
Висновок через ДІ: не відхиляємо H₀.


### Змістовне тлумачення помилок

- **Помилка I роду:** ми відхилили б H₀ і заявили, що стара норма Києва `20.0 °C` уже не відповідає дійсності, хоча насправді середня липнева температура все ще дорівнює `20.0 °C`. Для кліматологів це означало б необґрунтовано оновлений офіційний довідник. Ймовірність такої помилки контролюється рівнем `α = 0.05`.
- **Помилка II роду:** ми не відхилили б H₀ і залишили стару норму, хоча справжня середня липнева температура Києва вже змінилася. Тоді офіційний довідник пропустив би реальну кліматичну зміну. Це не та сама помилка: тут проблема полягає в пропущеній зміні, а не в хибному оголошенні зміни.

Для кліматологів дорожчою може бути помилка II роду: пропущена стійка зміна призведе до тривалого використання застарілої норми. Водночас помилка I роду також має ціну, бо випадковий результат однієї вибірки може спричинити передчасне оновлення норми.

## Контрольні питання

**1. Що означає p-value?** `p-value` — це ймовірність отримати за істинної H₀ результат, такий самий або ще більш екстремальний щодо `μ₀`, ніж спостережений. Це **не** ймовірність того, що H₀ правильна: p-value обчислюється за припущення, що H₀ уже істинна, і не є `P(H₀ | дані)`.

**2. Чому тип тесту обирають заздалегідь?** Якщо вибрати однобічний або двобічний тест після перегляду `x̄`, можна штучно підібрати вигідніший хвіст і збільшити частоту хибного відхилення H₀. Напрямок альтернативи має випливати з питання до аналізу даних.

**3. Чому помилки I і II роду різні?** Помилка I роду — хибна тривога: ми оголосили зміну, якої немає. Помилка II роду — пропуск: ми залишили стару норму, хоча зміна є. У контексті Києва це відповідно передчасне оновлення довідника та збереження застарілої норми.

**4. Чи узгодилися тест та інтервал?** Так: перевірка `μ₀` на належність до 95%-го інтервалу дала те саме рішення, що й двобічний тест при `α = 0.05`. Це випливає з еквівалентності критеріїв.

## Підсумок

- Для Києва згенеровано одну вибірку з 30 денних температур (`seed=1`, центр `21 °C`).
- Сформульовано двобічні H₀ і H₁ до обчислення p-value.
- Через `scipy.stats.norm.cdf()` обчислено z-статистику та p-value.
- Рішення прийнято при `α = 0.05`, а висновок перевірено через 95%-й довірчий інтервал.
- Помилки I та II роду пояснено в контексті старої кліматичної норми Києва.